In [ ]:
import os

import requests
from dotenv import load_dotenv
import time
import csv


load_dotenv(".env")
api_key_TMDB = os.getenv("TMDB_API_KEY")


if not api_key_TMDB:
    print("API KEY not found check env file")
    exit()
else:
    print("TMDB KEY loaded")



#OMDb key
load_dotenv(".env")
api_key_OMDB = os.getenv("OMDB_API_KEY")


if not api_key_OMDB:
    print("API KEY not found check env file")
    exit()
else:
        print("OMDB KEY loaded")

TMDB KEY loaded
OMDB KEY loaded


# Frage 10
Build Api accesslink + get data



In [ ]:



def load_movies_tmdb(api_key, start_date="2010-01-01", end_date="2020-12-31", genre_id="28", min_page=1, max_page=2):
    """
    Loads movies from a start to end date and based on genre
    28= action , 12 = Adventure, 35 = Comedy, 18 = Drama, 14 Fantasy , 27 = Horror
    """
   
    # Result list
    all_top_movies = []
    base_url = "https://api.themoviedb.org/3"
    discover_url = f"{base_url}/discover/movie"
   
    
    
    print(f"Start download from {start_date} to {end_date}From page {min_page} to {max_page}")
    
    
    for actual_page in range(min_page, max_page + 1):
        
       
        # Filter
        parameter = {
            "api_key": api_key,
            "primary_release_date.gte": start_date,  
            "primary_release_date.lte": end_date,    
            "with_genres": genre_id,                 
            "language": "de-DE",
            "sort_by": "revenue.desc",
            "page": actual_page,
            "vote_average.gte": 0.1,
            "vote_count.gte": 50
        }
        
        
        # Send api request
        response = requests.get(discover_url, params=parameter)
        
        if response.status_code == 200:
            data = response.json()
            if actual_page == min_page:
                print(f"Total movies found according to criteria:{data.get('total_results')} on total pages: {data.get('total_pages')}")
            movies_on_page = data.get("results", [])
            
            # For each film per page
            for movie in movies_on_page:
                tmdb_id = movie.get("id")
                imdb_id = "Not found"
                revenue = 0
                
                # request link to get imdb id
                detail_url = f"{base_url}/movie/{tmdb_id}"
                detail_params = {"api_key": api_key}
                
                # send request
                try:
                    detail_response = requests.get(detail_url, params=detail_params)
                    if detail_response.status_code == 200:
                        detail_data = detail_response.json()
                        imdb_id = detail_data.get("imdb_id", "Not found")
                        revenue = detail_data.get("revenue", 0)
                except:
                    pass
                
                movie["imdb_id"] = imdb_id
                movie["revenue"] = revenue
                
                # add movie to result
                all_top_movies.append(movie)
                
                # Spam protection
                time.sleep(0.05)
                
            print(f"Loaded page {actual_page}.")
            
        else:
            print(f"Error on page {actual_page}: Code {response.status_code}")
            break
            
        # Spam protection for pages
        time.sleep(0.3)
        
    print(f"Finished {len(all_top_movies)} movies ")
    
    return all_top_movies
    

In [ ]:
#OMDb get block
def add_omdb_details(movies_list, omdb_api_key):
    """
    Takes list from tmdb with imdb id and adds plot to list
    """
    
    print("Start")
    
    
    base_url = "http://www.omdbapi.com/"
    
    # Loop for each movie
    for index, movie in enumerate(movies_list):
        
        imdb_id = movie.get("imdb_id")
        
        # CHECK VALID ID
        if not imdb_id or imdb_id == "Not found":
            movie["plot"] = "No IMDb ID "
            continue
            
        
        # i iss used for imdb id , plot = full or short
        params = {
            "apikey": omdb_api_key,
            "i": imdb_id,
            "plot": "full" 
        }
        
        # Send request 
        try:
            response = requests.get(base_url, params=params)
            
            if response.status_code == 200:
                data = response.json()
                
                # Check id exist
                if data.get("Response") == "True":
                    # Extract the plot and add it to our movie dictionary
                    movie["plot"] = data.get("Plot", "No plot")
                    movie["genre"] = data.get("Genre", "No genre")
                    movie["box_office"] = data.get("BoxOffice", "0")
                else:
                    movie["plot"] = "Movie plot not on OMDb"
                    movie["genre"] = "Movie genre not on OMDb"
                    movie["box_office"] = "Movie box office not on OMDb"
            else:
                movie["plot"] = f"plot API Error: {response.status_code}"
                movie["genre"] = f"genre API Error: {response.status_code}"
                movie["box_office"] = f"box_office API Error: {response.status_code}"
                
        except Exception as e:
            movie["plot"] = "Connection failed plot"
            movie["genre"] = "Connection failed genre"
            movie["box_office"] = "Connection failed box office"
            
            
            
        # Progresss check
        if (index + 1) % 10 == 0:
            print(f" plots for {index + 1} movies")
            
        
        time.sleep(0.15)
        
    print("finished adding plots from OMDb")
    
    return movies_list

In [ ]:


def save_movies_to_csv(movie_list, filename="movies_from_year_2010.csv"):
    """
    Saves datafram into csv to allowing caching of result
    """
    
    print(f"Save data to {filename}")
    
   
    file_exists = os.path.isfile(filename)
    
    
    with open(filename, mode='a', newline='', encoding='utf-8') as file:
        
        # column names 
        columns = ['id', 'title', 'rating', 'release_date', 'original_language', 'IMDb_ID', 'genre', 'plot','box_office', 'revenue']
        writer = csv.DictWriter(file, fieldnames=columns)
        
       
        if not file_exists:
            writer.writeheader()
            
        # Write movies as rows
        for index, movie in enumerate(movie_list, start=1):
            
            # Map the dictionary keys to csv colums
            writer.writerow({
                'id': index,  
                'title': movie.get('title', 'N/A'),
                'rating': movie.get('vote_average', 'N/A'),
                'release_date': movie.get('release_date', 'N/A'),
                'original_language': movie.get('original_language', 'N/A'),
                'IMDb_ID': movie.get('imdb_id', 'Not found'),
                'genre': movie.get('genre', 'Not found'),   
                'plot': movie.get('plot', 'Not found'),  
                'box_office': movie.get('box_office', 0),
                'revenue': movie.get('revenue', 0)    
            })
            
    print("Finished")

In [ ]:
#testblock
movies_2010=load_movies_tmdb(api_key_TMDB,"2010-01-01","20220-12-31","28",1,2)
movies_2010_deatils = add_omdb_details(movies_2010,api_key_OMDB)
save_movies_to_csv(movie_list=movies_2010_deatils, filename="movies_plot.csv")


Start download from 2010-01-01 to 20220-12-31From page 80 to 82
Total movies found according to criteria:3232 on total pages: 162
Loaded page 80.
Loaded page 81.
Loaded page 82.
Finished 60 movies 
Start
 plots for 10 movies
 plots for 20 movies
 plots for 30 movies
 plots for 40 movies
 plots for 50 movies
 plots for 60 movies
finished adding plots from OMDb
Save data to movies_plot.csv
Finished
